# A1.15 · Overwhelming the human in the loop

**Function A — Securing AI Architectures → CyberTravels' Architecture, and Every Risk It Carries**  ·  *Security of AI*

Builds on **[A1.14 · Repudiation and untraceability](https://spbreed.github.io/cyber-commons/lessons/A1.14.html)**.

| | |
|---|---|
| Tools used | standard library only |

## What this lesson is

**What it covers.** Push approval volume up and measure the point at which review quality collapses.

**Why a security engineer needs it.** The approval gate is recorded as a control and operates as a click. At volume it approves everything, including the one request that mattered. The control it builds is: approval reserved for irreversible actions, with everything else bounded by policy (A3.6).

This is a **risk** lesson: it shows the failure happening before anything tries to stop it, so the control that follows is answering something you have already watched go wrong.

## 1 · The hook

Approval is a genuine control at four requests a day. At four hundred it is a person clicking approve, and the control has quietly become a log of things somebody scrolled past.

> **At CyberTravels.** Approval on every refund is a real control at four a day. CyberTravels generates four hundred, and the control quietly becomes a log of things somebody scrolled past. R2.

## 2 · The framework

```
   requests/hour     4        40       400
   read carefully   yes      some      no
   approval is    control  friction  a log

   the control does not fail loudly. it degrades into a click.
```

**OWASP T10 — Overwhelming Human-in-the-Loop.**

Human approval is the control everyone reaches for first. It is placed at the
tool call — the right place — and it is genuinely strong for rare, consequential
decisions.

Then the system scales, and the arithmetic turns on it.

An agent generates approval requests at machine speed. A human reads them at
human speed. When the queue exceeds what a person can actually consider, the
behaviour does not degrade gracefully into "slower but careful". It degrades
into **approving without reading**, because the alternative is being the reason
nothing shipped.

The failure is invisible from inside the system. Every approval is recorded. The
audit trail shows a human decision on every action. The control appears to be
operating at 100%, and the thing being measured — that a human clicked — is not
the thing anyone cared about.

There is a second-order effect worth naming: an attacker who wants one approval
can *manufacture the volume that makes it likely*. Generate two hundred benign
requests, put the one that matters at position 173, and the control has been
defeated by arithmetic rather than by cleverness.

Approval is a control for irreversible actions. Used as a general-purpose gate
it becomes a click, and the risk register still counts it.

> **Where this lands on the reference architecture.**
>
> ```
> ingress -> orchestrator -> agent_runtime -> model
>                                |              |
>                          messaging        tools / mcp
>                                |              |
>                       knowledge / memory   egress
>            identity + policy wrap every call · observability records it
> ```

## 3 · The risk, realised

Approval quality against volume, and the position an attacker chooses.

## 4 · The check, as a skill

Approval coverage at CyberTravels reads 100% at every volume, because coverage measures whether a human was asked. The skill models what reading does instead, and finds the volume at which the gate stops being one.

In [ ]:
# skills/threats/approval-queue-saturation-model/SKILL.md — embedded verbatim from the repository.
# This is the file itself, not a paraphrase of it.
SKILL_MD = r"""---
name: approval-queue-saturation-model
description: >-
  Model what happens to an approval gate as volume rises — coverage staying at
  100% while actual review collapses — and find the volume at which the control
  stops working. Use when reviewing human-in-the-loop design, approval fatigue,
  or a gate that has never been measured.
allowed-tools: Read, Grep, Glob
---

# Coverage stays at 100%; reading does not

An approval gate reports the same number at every volume, because coverage
measures whether a human was *asked*. What degrades is whether they read it,
and an attacker who can choose position only has to generate enough requests to
sit behind.

## When to use this

Any control whose enforcement is a person: approvals, exception reviews, alert
triage sign-off, change advisory.

## Procedure

**1 — Measure current volume,** per reviewer per day. Not the design volume —
the observed one, at peak rather than mean.

**2 — Establish the reading budget.** How many items can one reviewer consider
properly in a shift? Ask them; the number is usually between 20 and 30 and it
is always far below the queue.

**3 — Model detection against position.** Place a malicious item at various
depths and compute the probability it is actually read. The curve falls off a
cliff at the reading budget, not gradually.

**4 — Note who controls position.** If a requester can generate the items in
front of theirs, depth is attacker-chosen and the average case is irrelevant.

**5 — Report the two numbers that change the design.** Items per reviewer per
day, and the reading budget. The gap between them is the finding, and it points
at routing by reversibility rather than at hiring.

## Output contract

```json
{
  "volume": {"per_reviewer_per_day": 0, "measured_at": "peak|mean"},
  "reading_budget": 0,
  "coverage_reported": 1.0,
  "detection_by_depth": [{"depth": 0, "read_probability": 0.0}],
  "position_attacker_controlled": true,
  "gap": 0
}
```

## Failure modes

- **Reporting coverage.** It is 100% by construction and means nothing.
- **Using mean volume.** The gate fails at peak.
- **Recommending more reviewers.** The fix is fewer items, chosen by
  reversibility.
"""

In [ ]:
import json, re

def parse_skill(md):
    """Split a SKILL.md into (frontmatter dict, body).

    Frontmatter is a small, fixed subset of YAML: `key: value`, plus folded
    scalars (`description: >-`) whose continuation lines are indented. That is
    all a skill needs, and parsing it directly means no dependency.
    """
    if not md.startswith("---"):
        raise ValueError("a SKILL.md must open with a frontmatter block")
    _, front, body = md.split("---", 2)
    meta, key = {}, None
    for line in front.strip().splitlines():
        if not line.strip():
            continue
        if not line[0].isspace() and ":" in line:
            key, val = line.split(":", 1)
            key, val = key.strip(), val.strip()
            # `>-` and `|` open a folded block; the value is on the next lines
            meta[key] = "" if val in (">-", ">", "|", "|-") else val
        elif key is not None:
            meta[key] = (meta[key] + " " + line.strip()).strip()
    if "allowed-tools" in meta:
        meta["allowed-tools"] = [t.strip() for t in meta["allowed-tools"].split(",")
                                 if t.strip()]
    for required in ("name", "description"):
        if not meta.get(required):
            raise ValueError(f"skill is missing a {required!r}")
    return meta, body.strip()

_WORD = re.compile(r"[a-z][a-z-]{3,}")

def route(task, skills):
    """Pick the skill whose description best matches a task. Deterministic.

    The description is not documentation — it is the routing key. An agent
    decides whether to load a skill by reading it, so a vague description means
    the skill never fires when it should, and two overlapping descriptions mean
    the wrong one fires.

    Returns (pick, scores, margin). A margin of 0 means the top two scored the
    same and the "winner" is just whichever sorted first — an arbitrary answer
    wearing a confident face. Callers should refuse to auto-route on margin 0
    rather than pretend the tiebreak meant something.
    """
    want = set(_WORD.findall(task.lower()))
    def score(meta):
        return len(want & set(_WORD.findall(meta["description"].lower())))
    scores = {n: score(skills[n]) for n in sorted(skills)}
    # sort names first, then by score: ties must break identically on every
    # machine or the same task routes differently on two runs
    ranked = sorted(sorted(skills), key=lambda n: -scores[n])
    top = scores[ranked[0]]
    margin = top - (scores[ranked[1]] if len(ranked) > 1 else 0)
    return ranked[0], scores, margin

def contract_of(body):
    """The JSON block under '## Output contract' — the skill's machine promise."""
    # non-greedy across any prose between the heading and the fence
    m = re.search(r"## Output contract\b.*?```json\n(.*?)```", body, re.S)
    if not m:
        raise ValueError("skill declares no output contract")
    return json.loads(m.group(1))

def check(instance, contract, path="$"):
    """Structural conformance of an instance against a contract template.

    Returns the list of problems. An empty list means the shape is right — and
    that is *all* it means. Conformance is not accuracy: an empty findings list
    conforms perfectly and tells you nothing.
    """
    problems = []
    if isinstance(contract, dict):
        if not isinstance(instance, dict):
            return [f"{path}: expected an object, got {type(instance).__name__}"]
        for k, v in sorted(contract.items()):
            if k not in instance:
                problems.append(f"{path}.{k}: missing")
            else:
                problems += check(instance[k], v, f"{path}.{k}")
    elif isinstance(contract, list):
        if not isinstance(instance, list):
            return [f"{path}: expected a list, got {type(instance).__name__}"]
        for i, item in enumerate(instance):          # every element, same template
            problems += check(item, contract[0], f"{path}[{i}]")
    elif isinstance(contract, str) and "|" in contract:
        if instance not in contract.split("|"):
            problems.append(f"{path}: {instance!r} is not one of {contract}")
    elif isinstance(contract, bool):                  # before the numeric case:
        if not isinstance(instance, bool):            # bool is a subclass of int
            problems.append(f"{path}: expected bool, got {type(instance).__name__}")
    elif isinstance(contract, (int, float)):
        # JSON has one number type. A contract written `0` must accept 0.4, or
        # every cost and rate in the pipeline has to be rounded to satisfy a
        # checker rather than to be correct.
        if isinstance(instance, bool) or not isinstance(instance, (int, float)):
            problems.append(f"{path}: expected a number, got {type(instance).__name__}")
    elif not isinstance(instance, type(contract)):
        problems.append(f"{path}: expected {type(contract).__name__}, "
                        f"got {type(instance).__name__}")
    return problems

# Execute the skill above: parse skills/threats/approval-queue-saturation-model/SKILL.md into the two
# halves an agent uses — the frontmatter it routes on, and the body
# it follows.
meta, body = parse_skill(SKILL_MD)
print(f"loaded skill: {meta['name']}")
print(f"  tools it may use: {', '.join(meta.get('allowed-tools', [])) or '—'}")
print(f"  routing description: {len(meta['description'].split())} words")
print(f"  procedure: {len(body.splitlines())} lines")

In [ ]:
# skills/threats/approval-queue-saturation-model/scripts/approval_queue_saturation_model.py — embedded verbatim from the repository.
# This is the skill's own script, not a paraphrase of it.
#!/usr/bin/env python3
"""Model what approval coverage and actual review both do as the queue grows.

This is the executable half of the `approval-queue-saturation-model` skill: the check the
SKILL.md next to it describes, run against a synthetic CyberTravels
estate so two runs can be diffed and the result argued with.

Standard library only, and deterministic, so it runs on a Kaggle
kernel with the internet switched off.
"""

def review(queue, careful_capacity=25):
    """A reviewer considers the first `careful_capacity` properly. The rest are
    approved to keep the queue moving - which is what actually happens."""
    considered = queue[:careful_capacity]
    rubber_stamped = queue[careful_capacity:]
    caught = [r for r in considered if r["malicious"]]
    missed = [r for r in rubber_stamped if r["malicious"]]
    return {"considered": len(considered), "rubber_stamped": len(rubber_stamped),
            "caught": len(caught), "missed": len(missed)}

def build_queue(volume, malicious_at):
    return [{"id": i, "malicious": i == malicious_at} for i in range(volume)]

print(f"{'daily volume':>13}{'considered':>12}{'stamped':>9}{'caught':>8}{'missed':>8}")
for volume in (10, 25, 100, 500):
    q = build_queue(volume, malicious_at=volume - 1)     # attacker picks the position
    r = review(q)
    print(f"{volume:>13}{r['considered']:>12}{r['rubber_stamped']:>9}"
          f"{r['caught']:>8}{r['missed']:>8}")

print()
print("At every volume the audit trail shows a human approval on 100% of")
print("actions. The control reports full coverage in all four rows.")
print()
print("The attacker does not need to defeat the reviewer. They need to arrive")
print("at position 173 of 200, and generating positions 1 to 172 is free.")
r = review(build_queue(500, malicious_at=499))
assert r["missed"] == 1 and r["caught"] == 0

## What you just proved

Approval coverage reads 100% at every volume while the malicious request is caught only when the queue is small enough to be read — and an attacker choosing the position needs only to generate the requests in front of it.

## Your turn

Count how many approval requests one of your agents generates per day and ask the person approving them how many they read in full. The gap between those two numbers is the control's real coverage.

---

**Next → [A1.16 · Misaligned and deceptive behaviour](https://spbreed.github.io/cyber-commons/lessons/A1.16.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/A1.15.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/A1.15.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*